# 02 — Fine-tuning et comparaison avec le modèle de base (P2 → P3)

Enchaîne tout le pipeline sur le jeu complet *Fashion Product Images (Small)* :

1. préparation des données (P1) ;
2. modèle de base : embeddings ResNet50 ImageNet + évaluation (P2) ;
3. fine-tuning CosFace sur `articleType` (P3) ;
4. comparaison avec / sans fine-tuning sur le split `test`.

À exécuter sur **Google Colab** ou **Kaggle** avec un GPU (fine-tuning de ResNet50 : quelques minutes par époque sur T4). En local sur CPU, utiliser `mobilenet_v3_large` et `--steps-per-epoch` pour un essai.

In [ ]:
# Sur Colab / Kaggle : récupérer le code et les dépendances
# !git clone https://github.com/monssefbaakka/stylelens.git
# %cd stylelens/ml
# !pip install -q -r requirements.txt
import os
from pathlib import Path

assert Path('stylelens').is_dir(), 'Lancer ce notebook depuis ml/ (ou faire %cd stylelens/ml)'
BACKBONE = 'resnet50'          # ou efficientnet_b0, mobilenet_v3_large
DATA_DIR = Path('data/fashion-product-images-small')

In [ ]:
# Données : token Kaggle requis (~/.kaggle/kaggle.json ou variables KAGGLE_USERNAME / KAGGLE_KEY)
if not (DATA_DIR / 'styles.csv').exists():
    !kaggle datasets download -d paramaggarwal/fashion-product-images-small -p data --unzip
    # le zip contient styles.csv et images/ à la racine : on les range dans DATA_DIR
    if (Path('data') / 'styles.csv').exists():
        DATA_DIR.mkdir(parents=True, exist_ok=True)
        os.replace('data/styles.csv', DATA_DIR / 'styles.csv')
        os.replace('data/images', DATA_DIR / 'images')
print((DATA_DIR / 'styles.csv').exists(), len(list((DATA_DIR / 'images').glob('*.jpg'))), 'images')

## 1. Préparation des données (P1)

In [ ]:
!python -m stylelens.data --data-dir {DATA_DIR} --out data/processed

## 2. Modèle de base : embeddings ImageNet (P2)

In [ ]:
!python -m stylelens.embeddings --model {BACKBONE} --out data/embeddings/{BACKBONE}
!python -m stylelens.evaluate --index data/embeddings/{BACKBONE}

## 3. Fine-tuning (P3)

Backbone gelé pendant 2 époques, puis réseau complet avec un LR 10× plus faible pour le backbone (remontée progressive sur une époque, BatchNorm figées). Validation mAP@10 sur `val` à chaque époque, meilleur modèle conservé, arrêt anticipé après 4 époques sans progrès.

In [ ]:
!python -m stylelens.train --backbone {BACKBONE} --epochs 15 --freeze-epochs 2 --batch-size 64 --out models/{BACKBONE}-ft

In [ ]:
import json
import matplotlib.pyplot as plt
import pandas as pd

history = pd.DataFrame(json.loads(Path(f'models/{BACKBONE}-ft/history.json').read_text(encoding='utf-8')))
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
history.plot(x='epoch', y='loss', ax=axes[0], marker='o', title='Perte CosFace (train)')
history.plot(x='epoch', y=['val_precision@1', 'val_map@10'], ax=axes[1], marker='o', title='Recherche sur val')
plt.show()
history

## 4. Comparaison avec et sans fine-tuning (split test)

In [ ]:
!python -m stylelens.embeddings --checkpoint models/{BACKBONE}-ft/model.pt --out data/embeddings/{BACKBONE}-ft
!python -m stylelens.compare data/embeddings/{BACKBONE} data/embeddings/{BACKBONE}-ft --out data/comparaison.md
!python -m stylelens.compare data/embeddings/{BACKBONE} data/embeddings/{BACKBONE}-ft --filter masterCategory --out data/comparaison_filtre.md

In [ ]:
# Exemples de requêtes : modèle de base (ligne du haut) vs fine-tuné (ligne du bas)
import numpy as np
from PIL import Image
from stylelens.embeddings_io import load_index
from stylelens.search import knn_search

base, idx_b, _ = load_index(f'data/embeddings/{BACKBONE}')
ft, idx_f, _ = load_index(f'data/embeddings/{BACKBONE}-ft')
assert (idx_b['id'].to_numpy() == idx_f['id'].to_numpy()).all()
q_mask = (idx_b['split'] == 'test').to_numpy()
g_mask = idx_b['split'].isin(['train', 'val']).to_numpy()
gallery = idx_b[g_mask].reset_index(drop=True)
rng = np.random.default_rng(0)

for qi in rng.choice(np.flatnonzero(q_mask), size=4, replace=False):
    fig, axes = plt.subplots(2, 6, figsize=(13, 5))
    for row, emb in enumerate([base, ft]):
        nn_idx, sims = knn_search(emb[qi:qi + 1], emb[g_mask], 5)
        axes[row, 0].imshow(Image.open(DATA_DIR / idx_b.loc[qi, 'image_path']))
        axes[row, 0].set_title(f"requête\n{idx_b.loc[qi, 'articleType']}", fontsize=8)
        for ax, j, s in zip(axes[row, 1:], nn_idx[0], sims[0]):
            hit = gallery.loc[j, 'articleType']
            ax.imshow(Image.open(DATA_DIR / gallery.loc[j, 'image_path']))
            ax.set_title(f'{hit}\n{s:.3f}', fontsize=8, color='green' if hit == idx_b.loc[qi, 'articleType'] else 'red')
        for ax in axes[row]:
            ax.axis('off')
    axes[0, 0].text(-0.3, 0.5, 'ImageNet', transform=axes[0, 0].transAxes, rotation=90, va='center')
    axes[1, 0].text(-0.3, 0.5, 'fine-tuné', transform=axes[1, 0].transAxes, rotation=90, va='center')
    plt.show()